In [1]:
'''
Evaluate the performance of search algorithms 
Collect scores across all prompts and trials, and compute the overall statistics.
'''

import os, psutil, gc
import time 
import json
import pprint

from collections import defaultdict
import random

import numpy as np
np.set_printoptions(precision=4)
from scipy.stats import ttest_rel

In [2]:
from sal.config import Config

from datasets import load_dataset

# from core import best_of_n
from utils.load_data import load_data_prm800k
from utils import grader 
from utils import grader2
from utils import parser

In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_tokenizer_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_tokenizer_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"

In [4]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)


# ds_completions = load_completions(completions_dir)

# load random_seeds     
# random_seeds = np.loadtxt("random_seeds.txt").astype("int64")
# random_seeds = [int(seed) for seed in random_seeds]

1: 43
2: 90
3: 105
4: 128
5: 134


In [5]:
import signal

In [11]:
num_trials = 2
num_questions = 128
metric_name = "naive1b"

config_names_arr = [
    "bob--v11--n-8--d-40--level-4",
    "mcts--v12--n-8--d-40--cpuct-2--level-4",
    "mcts--v51--n-8--d-40--nb-1--lam-10--dalpha-0.1--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4",
]

# config_names_arr = [
#     "bob--v11--n-8--d-40--level-4",
#     "bob--v11--n-40--d-40--level-4",
#     "mcts--v12--n-8--d-40--cpuct-2--level-4",
#     "mcts--v12--n-8--d-40--nb-5--cpuct-2--level-4",
#     "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-0.0--dbeta-1.0--cpuct-0-0--ppl-True--normalize-True--level-4",
#     "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-0.1--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4",
#     "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-10.0--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4",
#     "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-100.0--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4",
#     "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-1.0--dbeta-0.0--cpuct-0-0--ppl-True--normalize-True--level-4",
# ]

# config_names_arr = [
#     "bob--v11--n-8--d-40--level-4",
#     "sdp--v11--n-8--bw-4--d-40--lam-10--True--dalpha-0--dbeta-1.0--ppl-True--level-4",
#     "sdp--v11--n-8--bw-4--d-40--lam-10--True--dalpha-10--dbeta-1.0--ppl-True--level-4",
#     "sdp--v11--n-8--bw-4--d-40--lam-10--True--dalpha-100--dbeta-1.0--ppl-True--level-4",
#     "sdp--v11--n-8--bw-4--d-40--lam-10--True--dalpha-1.0--dbeta-0--ppl-True--level-4",
#     "mcts--v12--n-8--d-40--cpuct-2--level-4",
#     "mcts--v51--n-8--d-40--nb-1--lam-10--dalpha-0.1--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4",
# ]

nlen = len(config_names_arr)


results = []
for cidx in range(nlen):
    config_name = config_names_arr[cidx]
    print(f"\n")
    print(config_name)
    
    try:
        
        res = np.loadtxt(f"results/{config_name}/{metric_name}_{config_name}.txt")
        # results.append(res[:num_trials*128])
        config_scores = []
        for tidx in range(num_trials):
            config_scores.append(res[tidx*num_questions:(tidx+1)*num_questions])
        print(len(config_scores))
        config_scores = np.stack(config_scores, axis=1)
        config_means = np.mean(config_scores, axis=1)
        config_easy_idxes = np.where(config_means == 1.0)[0]
        config_hard_idxes = np.where(config_means == 0.0)[0]
        print(len(config_easy_idxes), config_easy_idxes)
        print(len(config_hard_idxes), config_hard_idxes)

        res_mean = np.mean(res)
        res_std = np.std(res, ddof=1)/np.sqrt(num_trials*128)
        print(f"mean = {res_mean:0.4f} (\u00B1{res_std:0.4f})")
        
        
    except Exception as e:
        
        print(f"An unexpected error occurred: {e}")
        res = None
        results.append(res)
    
        



bob--v11--n-8--d-40--level-4
2
28 [  7  12  14  18  21  27  30  32  37  39  55  57  58  64  69  83  87  90
  92  95 101 105 108 109 112 117 118 121]
79 [  0   2   3   4   5   8   9  11  13  16  17  19  20  22  23  25  28  29
  31  33  36  40  41  42  43  44  46  47  48  49  50  51  52  53  54  56
  59  60  61  63  65  66  67  68  71  73  74  75  76  77  78  79  80  81
  82  84  86  88  89  91  93  94  97  98  99 100 103 104 106 107 113 114
 115 116 120 122 123 125 126]
mean = 0.3234 (±0.0293)


mcts--v12--n-8--d-40--cpuct-2--level-4
2
46 [  1   2   3   7  10  14  15  18  21  26  27  30  31  32  35  37  39  41
  43  44  48  49  55  57  58  64  66  69  70  73  83  87  90  92  95 101
 102 105 108 109 117 118 119 121 123 124]
53 [  0   6   9  11  13  16  22  23  25  28  29  33  40  42  46  47  50  52
  53  56  59  60  61  62  65  67  68  74  76  77  79  80  82  85  86  93
  94  97  98  99 103 104 106 107 110 111 113 115 116 122 125 126 127]
mean = 0.4727 (±0.0313)


mcts--v51--n-8--d-40-

In [37]:
num_trials = 2
num_questions = 128
metric_name = "pass1b"

bob_config = "bob--v11--n-40--d-40--level-4"
bob_res = np.loadtxt(f"results/{bob_config}/{metric_name}_{bob_config}.txt")
# results.append(res[:num_trials*128])
bob_scores = []
for tidx in range(num_trials):
    bob_scores.append(bob_res[tidx*num_questions:(tidx+1)*num_questions])
bob_scores = np.stack(bob_scores, axis=1)
bob_means = np.mean(bob_scores, axis=1)
bob_easy_idxes = set(np.where(bob_means == 1.0)[0])
# bob_hard_idxes = set(np.where(bob_means == 0.0)[0])
print(f"\n")
print(bob_config)
print(len(bob_easy_idxes), bob_easy_idxes)
# print(len(bob_hard_idxes), bob_hard_idxes)

mcts_config = "mcts--v12--n-8--d-40--nb-5--cpuct-2--level-4"
mcts_res = np.loadtxt(f"results/{mcts_config}/{metric_name}_{mcts_config}.txt")
mcts_scores = []
for tidx in range(num_trials):
    mcts_scores.append(mcts_res[tidx*num_questions:(tidx+1)*num_questions])
mcts_scores = np.stack(mcts_scores, axis=1)
mcts_means = np.mean(mcts_scores, axis=1)
mcts_easy_idxes = set(np.where(mcts_means == 1.0)[0])
# mcts_hard_idxes = np.where(mcts_means == 0.0)[0]

print(f"\n")
print(mcts_config)
print(len(mcts_easy_idxes), mcts_easy_idxes)

# mcts_easy_idxes = mcts_easy_idxes - bob_easy_idxes
# print(len(mcts_easy_idxes), mcts_easy_idxes)
# print(len(mcts_hard_idxes), mcts_hard_idxes)

mctsdiv_config = "mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-10.0--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4"
mctsdiv_res = np.loadtxt(f"results/{mctsdiv_config}/{metric_name}_{mctsdiv_config}.txt")
mctsdiv_scores = []
for tidx in range(num_trials):
    mctsdiv_scores.append(mctsdiv_res[tidx*num_questions:(tidx+1)*num_questions])
mctsdiv_scores = np.stack(mctsdiv_scores, axis=1)
mctsdiv_means = np.mean(mctsdiv_scores, axis=1)
mctsdiv_easy_idxes = set(np.where(mctsdiv_means == 1.0)[0])
# mctsdiv_hard_idxes = set(np.where(mctsdiv_means == 0.0)[0])

print(f"\n")
print(mctsdiv_config)
print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)

# print(f"\n")
# print(f"common_easy_idxes")
# common_easy_idxes = bob_easy_idxes & mcts_easy_idxes & mctsdiv_easy_idxes
# print(len(common_easy_idxes), common_easy_idxes)

# print(f"\n")
# print(mcts_config)
# mcts_easy_idxes = mcts_easy_idxes - common_easy_idxes
# print(len(mcts_easy_idxes), mcts_easy_idxes)

print(f"\n")
print(mctsdiv_config)
# mctsdiv_easy_idxes = mctsdiv_easy_idxes - common_easy_idxes
# print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)

mctsdiv_easy_idxes = mctsdiv_easy_idxes - mcts_easy_idxes
print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)





bob--v11--n-40--d-40--level-4
99 {0, 1, 2, 3, 5, 6, 7, 8, 10, 12, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 54, 55, 57, 58, 63, 64, 66, 67, 69, 70, 71, 72, 75, 78, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 100, 101, 102, 103, 105, 106, 107, 108, 109, 110, 111, 112, 114, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 127}


mcts--v12--n-8--d-40--nb-5--cpuct-2--level-4
83 {1, 2, 3, 4, 5, 7, 8, 10, 12, 14, 15, 17, 18, 20, 21, 22, 23, 24, 26, 27, 29, 30, 31, 32, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 47, 48, 49, 54, 55, 57, 58, 64, 66, 67, 69, 70, 72, 73, 78, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 91, 92, 93, 95, 96, 100, 101, 102, 103, 104, 105, 106, 108, 109, 112, 117, 118, 119, 121, 122, 123, 124, 125, 127}


mcts--v51--n-8--d-40--nb-5--lam-10--dalpha-10.0--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4
86 {1, 2, 3, 4, 5, 7, 8, 10, 12, 14, 15, 17, 18, 2

In [38]:
num_trials = 2
num_questions = 128
metric_name = "pass1b"

bob_config = "bob--v11--n-40--d-40--level-4"
bob_res = np.loadtxt(f"results/{bob_config}/{metric_name}_{bob_config}.txt")
# results.append(res[:num_trials*128])
bob_scores = []
for tidx in range(num_trials):
    bob_scores.append(bob_res[tidx*num_questions:(tidx+1)*num_questions])
bob_scores = np.stack(bob_scores, axis=1)
bob_means = np.mean(bob_scores, axis=1)
bob_easy_idxes = set(np.where(bob_means == 1.0)[0])
# bob_hard_idxes = set(np.where(bob_means == 0.0)[0])
print(f"\n")
print(bob_config)
print(len(bob_easy_idxes), bob_easy_idxes)
# print(len(bob_hard_idxes), bob_hard_idxes)

mcts_config = "mcts--v12--n-8--d-40--cpuct-2--level-4"
mcts_res = np.loadtxt(f"results/{mcts_config}/{metric_name}_{mcts_config}.txt")
mcts_scores = []
for tidx in range(num_trials):
    mcts_scores.append(mcts_res[tidx*num_questions:(tidx+1)*num_questions])
mcts_scores = np.stack(mcts_scores, axis=1)
mcts_means = np.mean(mcts_scores, axis=1)
mcts_easy_idxes = set(np.where(mcts_means == 1.0)[0])
# mcts_hard_idxes = np.where(mcts_means == 0.0)[0]

print(f"\n")
print(mcts_config)
print(len(mcts_easy_idxes), mcts_easy_idxes)

# mcts_easy_idxes = mcts_easy_idxes - bob_easy_idxes
# print(len(mcts_easy_idxes), mcts_easy_idxes)
# print(len(mcts_hard_idxes), mcts_hard_idxes)

mctsdiv_config = "mcts--v51--n-8--d-40--nb-1--lam-10--dalpha-0.1--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4"
mctsdiv_res = np.loadtxt(f"results/{mctsdiv_config}/{metric_name}_{mctsdiv_config}.txt")
mctsdiv_scores = []
for tidx in range(num_trials):
    mctsdiv_scores.append(mctsdiv_res[tidx*num_questions:(tidx+1)*num_questions])
mctsdiv_scores = np.stack(mctsdiv_scores, axis=1)
mctsdiv_means = np.mean(mctsdiv_scores, axis=1)
mctsdiv_easy_idxes = set(np.where(mctsdiv_means == 1.0)[0])
# mctsdiv_hard_idxes = set(np.where(mctsdiv_means == 0.0)[0])

print(f"\n")
print(mctsdiv_config)
print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)

# print(f"\n")
# print(f"common_easy_idxes")
# common_easy_idxes = bob_easy_idxes & mcts_easy_idxes & mctsdiv_easy_idxes
# print(len(common_easy_idxes), common_easy_idxes)

# print(f"\n")
# print(mcts_config)
# mcts_easy_idxes = mcts_easy_idxes - common_easy_idxes
# print(len(mcts_easy_idxes), mcts_easy_idxes)

print(f"\n")
print(mctsdiv_config)
# mctsdiv_easy_idxes = mctsdiv_easy_idxes - common_easy_idxes
# print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)

mctsdiv_easy_idxes = mctsdiv_easy_idxes - mcts_easy_idxes
print(len(mctsdiv_easy_idxes), mctsdiv_easy_idxes)





bob--v11--n-40--d-40--level-4
99 {0, 1, 2, 3, 5, 6, 7, 8, 10, 12, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 54, 55, 57, 58, 63, 64, 66, 67, 69, 70, 71, 72, 75, 78, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 100, 101, 102, 103, 105, 106, 107, 108, 109, 110, 111, 112, 114, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 127}


mcts--v12--n-8--d-40--cpuct-2--level-4
60 {1, 2, 3, 5, 7, 10, 14, 15, 17, 18, 20, 21, 26, 27, 30, 31, 32, 35, 36, 37, 39, 41, 43, 44, 45, 48, 49, 55, 57, 58, 64, 66, 69, 70, 71, 72, 73, 75, 81, 83, 87, 89, 90, 91, 92, 93, 95, 100, 101, 102, 105, 108, 109, 117, 118, 119, 121, 123, 124, 125}


mcts--v51--n-8--d-40--nb-1--lam-10--dalpha-0.1--dbeta-1.0--cpuct-0-2--ppl-True--normalize-True--level-4
67 {1, 2, 3, 5, 7, 10, 14, 15, 17, 18, 20, 21, 24, 26, 27, 30, 31, 32, 35, 36, 37, 38, 39, 41, 42, 43, 44, 49, 55, 57, 58, 64, 66, 69, 70, 71, 72, 73, 75

In [ ]:
{96, 40, 42, 43, 81, 85, 24, 90, 127, 63}
{96, 5, 38, 42, 78, 84, 85, 24, 93, 127}

In [ ]:
{97, 34, 38, 71, 75, 114, 116, 63}
{96, 38, 103, 42, 78, 112, 82, 84, 85, 86, 24, 127}